# About
This notebook validates the structure of the German and English translations without validating the content. It spots missing sentences.

Note: The algorithms ignore different blank lines for now.

# Data import

In [3]:
import glob

# get all AsciiDoc files (hack: starting with "0", doesn't support more chapters > 9)
all_adoc_files = glob.glob('../docs/0*/*.adoc', recursive=True)

# filter out
# - "00-introduction" because they are just the files for combining AsciiDoc files
# - the reference list with books where translation validation makes no sense
file_list = [f for f in all_adoc_files if not f.endswith('00-introduction.adoc') and not f.endswith('00-references.adoc')]
file_list[:5] # just show first 5 items to not clutter the notebook with too much text

['../docs\\00-preamble\\01-what-to-expect.adoc',
 '../docs\\00-preamble\\02-out-of-scope.adoc',
 '../docs\\00-preamble\\03-prerequisites.adoc',
 '../docs\\00-preamble\\04-structure-timing-didactics.adoc',
 '../docs\\00-preamble\\05-exam-relevance-levels.adoc']

## Import files into a pandas DataFrame

In [12]:
import pandas as pd

from notebook import notebookapp
from IPython.display import display, HTML

import os

content_list = []

# make a Dataframe for each file
for file in file_list:

    # read in the AsciiDoc file, keep the blank lines for the line numbers
    df = pd.read_csv(file, names=['text'], sep="\r", skip_blank_lines=False)

    # Create a new 'lang' column initialized with empty values
    df['lang'] = ''

    # Mark the rows with language tags
    df.loc[df['text'] == '// tag::DE[]', 'lang'] = 'DE'
    df.loc[df['text'] == '// tag::EN[]', 'lang'] = 'EN'

    # Forward fill the 'lang' column to propagate the tags to the subsequent rows
    df['lang'] = df['lang'].replace('', pd.NA).ffill()

    # add line numbers in file
    df['line'] = df.index + 1
    
    # drop all blank lines which don't need to be checked
    df = df.dropna().copy()
    
    # just focus on texts that are relevant for translation checking
    df_lang = df[df['lang'].isin(['DE', 'EN'])]
    
    # separate translations
    df_de = df_lang[df_lang['lang'] == 'DE'].reset_index(drop=True)
    df_en = df_lang[df_lang['lang'] == 'EN'].reset_index(drop=True)
    
    # create a new DataFrame to get translations side by side
    content_df = pd.DataFrame({
        'text_DE': df_de['text'],
        'text_EN': df_en['text'],
        'line_DE': df_de['line'],
        'line_EN': df_en['line']

    })
     # add file name as first column
    content_df.insert(0,'filename',file)

    # completley optional but just to cool to not to do: create links that let you directly jump into your IDE
    # note: maybe the `replace` at the end needs to be adjusted to whatever IDE and operating system you have, but the default should work fine for most environments
    urls_DE = content_df.apply(lambda x: f"vscode://file/{os.path.abspath(x['filename'])}:{float(x['line_DE'])}".replace("/mnt/c/", "C:/"), axis=1)
    urls_EN = content_df.apply(lambda x: f"vscode://file/{os.path.abspath(x['filename'])}:{float(x['line_EN'])}".replace("/mnt/c/", "C:/"), axis=1)
    links_DE = urls_DE.apply(lambda x: f'<a href="{x}">DE</a>')
    links_EN = urls_EN.apply(lambda x: f'<a href="{x}">EN</a>')
    link_html = links_DE + " " + links_EN
    content_df.insert(1,'link',link_html)
    # drop all blank lines
    content_list.append(content_df.dropna().copy())

content = pd.concat(content_list, ignore_index=True)

# hint: remove the `.head()` method to get a complete list
display(HTML(content.head().to_html(escape=False)))

,filename,link,text_DE,text_EN,line_DE,line_EN
0,../docs\00-preamble\01-what-to-expect.adoc,DE EN,// tag::DE[],// tag::EN[],1.0,36.0
1,../docs\00-preamble\01-what-to-expect.adoc,DE EN,=== Was dieser Lehrplan enthält,=== What Does this Curriculum Contain,2.0,37.0
2,../docs\00-preamble\01-what-to-expect.adoc,DE EN,"Dieser Lehrplan für den _Certified Professional for Software Architecture - Foundation Level_ (CPSA-F) beinhaltet die Lernziele, die man beherrschen sollte, um die Rolle Softwarearchitekt:in zu übernehmen.",This curriculum for the Certified Professional for Software Architecture – Foundation Level (CPSA-F) outlines the essential learning goals that should be mastered to take up the role of software architect.,3.0,38.0
3,../docs\00-preamble\01-what-to-expect.adoc,DE EN,Seine Struktur orientiert sich an den grundlegenden Aktivitäten und Verantwortlichkeiten der Softwarearchitektur als Rolle:,It is structured along the fundamental activities and responsibilities of software architecture as a role:,5.0,40.0
4,../docs\00-preamble\01-what-to-expect.adoc,DE EN,* Anforderungen und Randbedingungen klären,* Clarifying stakeholder requirements and constraints,7.0,42.0


In [13]:
from openai import OpenAI
import os
import json

# set the role of the LLM
ai_system = """
You are a translator for iSAQB curricula between German and English.
Your task is to identify translation errors and inconsistencies between translations."
"""

# define the schema and format including what we expect from the LLM's result
response_json_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "result",
        "schema": {
            "type": "object",
            "properties": {
                "correctness": {
                    "type": "number",
                    "description" : "a value between 0 and 1 that indicates how well the two texts fit together"},
                "assessment": {
                    "type": "string",
                    "description" : "a brief assessment of how good the translation into English is"},
                "corrected_text_de": {
                    "type": "string",
                    "description" : "an improved translation of the German text"},
                "corrected_text_en": {
                        "type": "string",
                        "description": "an improved translation of the English text"}
            },
            "required": ["correctness", "assessment", "corrected_text_de", "corrected_text_en"],
            "additionalProperties": False
        },
        "strict": True
    }
}

openai_client = OpenAI(
   api_key = os.environ["OPENAI_API_KEY"]
)

# the magic function that does all the work for us!
def correct_translation(content):

    messages=[
        {"role": "system", "content": ai_system},
        {"role": "user", "content": content}
    ]

    completion = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages = messages,
        response_format = response_json_format,
        temperature = 0,
        n=1)

    return completion.choices[0]

In [19]:
import json

results = []

cache_filepath = "llm_results.json"
# Clear the file by opening it in write mode initially
with open(cache_filepath, 'w') as json_file:
    pass  # This clears the file


# Open the JSON file in append mode
with open(cache_filepath, 'a') as json_file:
    for i, r in content.iterrows():
        try:
            # Define the prompt for the LLM
            prompt = f"""
            Here is the German and English translation:

            German text:

            {r['text_DE']}

            English text:

            {r['text_EN']}

            Keep the AsciiDoc formatting as it is.

            Return a result in the defined JSON schema.
            """

            # Call the translation correction function
            result = correct_translation(prompt)

            # Convert the result to a Python dictionary
            result_dict = json.loads(result.message.content)

            # Append the result to the JSON file
            json.dump(result_dict, json_file)
            json_file.write('\n')  # Add a new line for each result

            # Optional: Immediately flush the file to ensure the result is written
            json_file.flush()

        except Exception as e:
            # Log the error and continue with the next row
            print(f"Error occurred on row {i}: {str(e)}")

result_df = pd.DataFrame.from_dict(results)
result_df.head()

KeyboardInterrupt: 

### Combine assessment result with content datam

In [15]:
assessed_content = part.join(result_df)[[
    'filename',
    'link',
    'text_DE',
    'corrected_text_de',
    'text_EN',
    'corrected_text_en',
    'correctness',
    'assessment',
    'line_DE',
    'line_EN']]

assessed_content.to_html("translation_assessment_report.html", escape=False)
assessed_content.to_html("translation_assessment_report.xslx", index=None, escape=False)
# hint: remove the `.head()` method to get a complete list
display(HTML(assessed_content.head().to_html(escape=False)))

,filename,link,text_DE,corrected_text_de,text_EN,corrected_text_en,correctness,assessment,line_DE,line_EN
0,../docs\00-preamble\01-what-to-expect.adoc,DE EN,// tag::DE[],// tag::DE[],// tag::EN[],// tag::EN[],1.0,"The translations are consistent and accurate, maintaining the original meaning and context.",1.0,36.0
1,../docs\00-preamble\01-what-to-expect.adoc,DE EN,=== Was dieser Lehrplan enthält,=== Was dieser Lehrplan enthält,=== What Does this Curriculum Contain,=== What this Curriculum Contains,0.9,"The translation is mostly accurate but the English version should use 'Curriculum' in a more consistent manner, as it is typically treated as a singular noun without 'Does'.",2.0,37.0
2,../docs\00-preamble\01-what-to-expect.adoc,DE EN,"Dieser Lehrplan für den _Certified Professional for Software Architecture - Foundation Level_ (CPSA-F) beinhaltet die Lernziele, die man beherrschen sollte, um die Rolle Softwarearchitekt:in zu übernehmen.","Dieser Lehrplan für den _Certified Professional for Software Architecture - Foundation Level_ (CPSA-F) beinhaltet die Lernziele, die man beherrschen sollte, um die Rolle der Softwarearchitekt:in zu übernehmen.",This curriculum for the Certified Professional for Software Architecture – Foundation Level (CPSA-F) outlines the essential learning goals that should be mastered to take up the role of software architect.,This curriculum for the Certified Professional for Software Architecture – Foundation Level (CPSA-F) includes the learning objectives that one should master to assume the role of software architect.,0.9,The translation is mostly accurate but could be improved for clarity and consistency in terminology.,3.0,38.0
3,../docs\00-preamble\01-what-to-expect.adoc,DE EN,Seine Struktur orientiert sich an den grundlegenden Aktivitäten und Verantwortlichkeiten der Softwarearchitektur als Rolle:,Seine Struktur orientiert sich an den grundlegenden Aktivitäten und Verantwortlichkeiten der Softwarearchitektur in ihrer Rolle:,It is structured along the fundamental activities and responsibilities of software architecture as a role:,It is structured along the fundamental activities and responsibilities of software architecture in its role:,0.9,"The translation is mostly accurate, but the phrase 'as a role' could be better integrated into the sentence for clarity.",5.0,40.0
4,../docs\00-preamble\01-what-to-expect.adoc,DE EN,* Anforderungen und Randbedingungen klären,* Anforderungen und Randbedingungen klären,* Clarifying stakeholder requirements and constraints,* Clarifying requirements and constraints,0.7,The translation captures the general meaning but introduces the term 'stakeholder' which is not present in the original German text. This could lead to a misunderstanding of the scope of the requirements being clarified.,7.0,42.0


### Correct metadata correctness values

In [17]:
# set metadata to 1 if OK
assessed_content.loc[
    (assessed_content['text_DE'] == "// tag::DE[]") &
    (assessed_content['text_EN'] == "// tag::EN[]"), 'correctness'] = 1
assessed_content.loc[
    (assessed_content['text_DE'] == "// end::DE[]") &
    (assessed_content['text_EN'] == "// end::EN[]"), 'correctness'] = 1

### Set the threshold
Set the threshold for correctness. The default with 0.8 may be too ambitious.m

In [18]:
THRESHOLD = 0.8
corrections = assessed_content[assessed_content['correctness'] < THRESHOLD]
display(HTML(corrections.to_html(escape=False)))

,filename,link,text_DE,corrected_text_de,text_EN,corrected_text_en,correctness,assessment,line_DE,line_EN
4,../docs\00-preamble\01-what-to-expect.adoc,DE EN,* Anforderungen und Randbedingungen klären,* Anforderungen und Randbedingungen klären,* Clarifying stakeholder requirements and constraints,* Clarifying requirements and constraints,0.7,The translation captures the general meaning but introduces the term 'stakeholder' which is not present in the original German text. This could lead to a misunderstanding of the scope of the requirements being clarified.,7.0,42.0


## Produce output files



### Diffable version
Version that produces a separate file to work with a diff editor. It's disabled by default via the `is_active` flag because the mergable version is way better (IMHO).

In [10]:
# this is jusis_active = False

# Group by 'filename' to process each file only once
for filename, group in corrections.groupby('filename'):
    # Open the file once for reading
    with open(filename, 'r') as file:
        lines = file.readlines()

    # Modify the relevant lines in memory
    modified = False
    for index, row in group.iterrows():
        line_number = int(row['line_EN']) - 1  # Convert to zero-based index
        print(filename)
        print(line_number)
        lines[line_number] = row['corrected_text_en'] + '\n'
        modified = True  # Mark as modified since we are replacing the line

    # Write the updated content to a new file ending with '_corrected.adoc'
    if modified and is_active:
        new_filename = filename.replace('.adoc', '_aicorrected.adoc')
        with open(new_filename, 'w') as file:
            file.writelines(lines)

### Mergeable version

Produces a merge file to work with a merge editor. If you're brave, you can also set `is_override` to true to merge the suggestions of the LLM directly into the original file.

Tip: Overriding only should be enabled if both sections of the translations are almost structurally identically because otherwise it leads to complete chaos!

**Warning: Before setting it to True, make sure that you committed your local changes! Good luck!**

In [11]:
# flag for setting the override of the original file
is_override = False

# Group correctiony for each file to process each file only once
for filename, group in corrections.groupby('filename'):
    # Open the file once for reading
    with open(filename, 'r') as file:
        lines = file.readlines()

    # Modify the relevant lines in memory
    modified = False
    for index, row in group.iterrows():
        if row['corrected_text_en']:  # Check if corrected_text_en is not empty
            line_number = int(row['line_EN']) - 1  # Convert to zero-based index
            # Insert merge-style corrections
            lines[line_number] = (
                f"<<<<<<< ORIGINAL\n{lines[line_number]}"
                f"=======\n{row['corrected_text_en']}\n"
                f">>>>>>> CORRECTED | German text: \'{row['text_DE']}\' (Line {int(row['line_DE'])}) | Assessment: {row['assessment']}\n"
            )
            modified = True  # Mark as modified since we are replacing the line

    # Write the updated content to a new file ending with '_merge.adoc'
    if modified:
        new_filename = filename if is_override else filename.replace('.adoc', '_merge.adoc')
        with open(new_filename, 'w') as file:
            file.writelines(lines)

        print(f"Merge-style corrected file saved as {new_filename}")